# VAR bayesianos (3): la pandemia

Con la muestra completa, hasta el último dato, el modelo macrofiscal del notebook B tiene que convivir
con 2020: el PBI cae 30 % interanual en 2020Q2 y rebota más de 40 % un año después. Aquí vemos qué le
hacen esos trimestres a la estimación y comparamos tres tratamientos:

- **sin tratamiento**: varianza constante;
- **dummies**: $\mathbf{y}_t = c + \Phi_1\mathbf{y}_{t-1} + \Phi_2\mathbf{y}_{t-2} + \Gamma\mathbf{d}_t + \mathbf{u}_t$,
  con un indicador por trimestre de 2020Q1 a 2021Q4 en $\mathbf{d}_t$;
- **escalamiento de volatilidad** (Lenza y Primiceri, 2022): $\mathbf{u}_t \sim N(0, s_t^2\Sigma)$, con
  $s_t$ estimado por verosimilitud marginal en la misma ventana y $s_t = 1$ fuera de ella.

*Tiempo de corrida: menos de un minuto.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MacroPy import BayesianVAR
from MacroPy.plots_uc import plot_fan_chart
from MacroPy.plots_kalman import set_bse_style
from macrofiscal_data import load_levels, yoy_growth, quarter

set_bse_style()
levels, source = load_levels()
growth = yoy_growth(levels)
names = ["x", "y", "r"]
col = {c: i for i, c in enumerate(names)}
n, p, h = 3, 2, 8
window = ("2020Q1", "2021Q4")
titles = {"x": "Precios de exportación (x)", "y": "PBI (y)",
          "r": "Ingresos fiscales (r)"}
priors = {"mn_mean": 0.5, "lambda1": 2, "lambda2": 1}

span = f"{quarter(growth.index[0])} a {quarter(growth.index[-1])}"
print(f"Fuente: {source}")
print(f"Muestra completa: {span} ({len(growth)} trimestres)")
growth.loc["2019-12-01":"2021-12-01"].round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)
for ax, c in zip(axes, names):
    ax.plot(growth.index, growth[c], color="#1F3D5C", lw=1.3)
    ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2021-12-31"),
               color="gray", alpha=0.18, lw=0)
    ax.axhline(0, color="gray", lw=0.7)
    ax.set_title(f"{titles[c]}, var. % interanual", fontsize=10)
fig.suptitle("Modelo macrofiscal con la pandemia: ventana 2020Q1-2021Q4 "
             "sombreada", fontsize=11, weight="bold")
plt.show()

## 1. Qué le hace la pandemia a la estimación

Estimamos el mismo modelo cuatro veces: hasta 2019Q4, que sirve de referencia, y con la muestra
completa bajo los tres tratamientos. Dos resúmenes por modelo, en medianas posteriores: la
**persistencia**, con el máximo módulo de los valores propios de $F$ y la suma de los dos rezagos
propios de cada ecuación, y la **desviación estándar de las innovaciones**, $\sqrt{\Sigma_{ii}}$.

In [ ]:
def estimate(sample, **covid):
    model = BayesianVAR(sample, lags=p, prior_type=2,
                        prior_params=priors,
                        post_draws=4000, burnin=0.5, fhor=h, seed=42,
                        **covid)
    model.sample_posterior()
    return model

models = {
    "hasta 2019Q4": estimate(growth.loc[:"2019-12-01"]),
    "sin tratamiento": estimate(growth),
    "dummies": estimate(growth, covid_mode="dummies",
                        covid_window=window),
    "Lenza-Primiceri": estimate(growth, covid_mode="lenza-primiceri",
                                covid_window=window),
}

def summary(model):
    """Posterior medians of persistence and innovation volatility."""
    k = model.ncoeff_eq
    coef = np.asarray(model.beta_draws).reshape(-1, n, k)
    sd = np.sqrt(np.array([np.diag(S) for S in model.Sigma_draws]))
    moduli = []
    for c in coef:                    # c: equations x regressors
        F = np.zeros((n * p, n * p))
        F[:n, :] = c[:, :n * p]       # [Phi_1, Phi_2]
        F[n:, :n] = np.eye(n)
        moduli.append(np.abs(np.linalg.eigvals(F)).max())
    row = {"máximo módulo de F": np.median(moduli)}
    for i, v in enumerate(names):
        own = coef[:, i, i] + coef[:, i, n + i]
        row[f"persistencia propia de {v}"] = np.median(own)
    for i, v in enumerate(names):
        row[f"desv. est. de u ({v})"] = np.median(sd[:, i])
    return row

scales = np.atleast_1d(models["Lenza-Primiceri"].covid_scales)
print("Escalas estimadas, 2020Q1 a 2021Q4 (s = 1 fuera de la ventana):",
      scales.round(2))
print("rho implícito:", round((scales[3] - 1) / (scales[2] - 1), 3))
pd.DataFrame({name: summary(m) for name, m in models.items()}).T.round(2)

Sin tratamiento la pandemia hace dos cosas, y ninguna es la que uno esperaría.

**Infla la varianza.** La desviación estándar de las innovaciones del PBI pasa de 1.54 a 5.43 puntos,
tres veces y media, y la de la recaudación de 6.06 a 9.36. Esa varianza inflada no se queda en 2020:
es un parámetro constante, así que contamina el horizonte entero.

**Distorsiona la dinámica.** La persistencia propia del PBI cae de 0.68 a 0.45, porque la caída de
2020 y el rebote de 2021 por efecto base se parecen a una serie que revierte rápido. Es el sesgo que
documenta Cascaldi-Garcia (2022). Los dos tratamientos la devuelven a 0.74 y 0.70, cerca de la
referencia.

Las escalas dicen cómo lo logra Lenza-Primiceri: 2020Q2 pesa como un dato con **diez veces** la
desviación estándar normal, y los trimestres siguientes con cerca de siete.

Una advertencia sobre la implementación: en `MacroPy` $\rho$ se estima solo con los trimestres de la
ventana, y sale 0.99. En el modelo original el decaimiento continúa después de 2021Q4, y los datos
posteriores, que se ven normales, empujarían $\rho$ hacia abajo.

## 2. Tres *fan charts*

Pronosticamos ocho trimestres desde el último dato con cada tratamiento. Las bandas van de 10 a
90 %, al estilo de los fan charts de banca central.

In [ ]:
last = growth.index[-1]
future = pd.date_range(last + pd.DateOffset(months=3), periods=h,
                       freq="3MS")
treatments = ["sin tratamiento", "dummies", "Lenza-Primiceri"]
fans = {name: models[name].forecast(fhor=h, plot_forecast=False)
        ["forecast_draws"] for name in treatments}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), constrained_layout=True,
                         sharey=True)
for i, (ax, name) in enumerate(zip(axes, treatments)):
    plot_fan_chart(growth["y"], fans[name][:, :, col["y"]], future,
                   bands=(0.90, 0.70, 0.50, 0.30, 0.10),
                   history_from="2017-03-01", hline=0, ax=ax,
                   color="#B0413E", center_label="mediana",
                   history_label="observado", title=f"PBI (y): {name}")
    ax.set_ylim(-25, 30)
    legend = ax.get_legend()
    if i != 1:
        legend.remove()                  # a single legend, in the middle
    else:
        legend.set_frame_on(True)
        legend.get_frame().set(facecolor="white", edgecolor="none",
                               alpha=0.95)
plt.show()

widths = {}
for name in treatments:
    end = fans[name][:, -1, :]
    widths[name] = {
        f"{v}: ancho {int(100 * cov)} % a 8 trimestres (pp)":
            np.percentile(end[:, col[v]], 50 + 50 * cov)
            - np.percentile(end[:, col[v]], 50 - 50 * cov)
        for v in ("y", "r") for cov in (0.90, 0.50)}
pd.DataFrame(widths).T.round(2)

Sin tratamiento, la banda al 90 % del PBI a ocho trimestres mide **22.5 puntos**, el doble que con
dummies (11.2) o con escalamiento (10.2). En la recaudación, 56.4 contra 41.8 y 41.0.

Conviene notar dónde está el exceso: en trimestres de 2026 y 2027, que no tienen nada que ver con la
pandemia. La varianza es un parámetro constante, así que el ruido de 2020 se paga en todo el
horizonte.

Entre dummies y Lenza-Primiceri la diferencia es pequeña, y tiene sentido: con el origen del
pronóstico fuera de la ventana y $s_t = 1$ desde entonces, ambos pronostican con la varianza de
tiempos normales. El argumento de Lenza y Primiceri contra descartar datos pesa sobre todo cuando se
pronostica **desde dentro** de la pandemia, con la volatilidad todavía alta: es el ejercicio 5.

## 3. ¿Y las respuestas al impulso?

Las bandas no son lo único contaminado. La **dinámica** también, y eso se ve en la respuesta a un
choque.

Usamos aquí el mismo factor de Cholesky $S$ con el que sorteamos los choques del fan chart: $\mathbf{u}_t =
S\varepsilon_t$ y $\Theta_j = \Psi_j S$. En el pronóstico esa elección no tenía contenido económico,
porque cualquier $S$ con $SS' = \Sigma$ da la misma densidad. Aquí sí lo tiene, y ese es justamente
el tema de la sesión 2. Por ahora lo tratamos como un ejercicio de diagnóstico: **si el mismo choque
produce respuestas distintas según el tratamiento, el problema está en la estimación, no en la
economía**.

In [ ]:
hor = 16
irfs = {}
for name in ["hasta 2019Q4"] + treatments:
    model = models[name]
    model.hor = hor + 1
    irfs[name] = model.compute_irfs()          # (extracciones, horizonte, variable, choque)

shock, response = col["x"], col["r"]           # choque externo -> ingresos fiscales
fig, ax = plt.subplots(figsize=(7.2, 3.6))
colors = {"hasta 2019Q4": "#1F3D5C", "sin tratamiento": "#B0413E",
          "dummies": "#5A7D43", "Lenza-Primiceri": "#9A7B2E"}
steps = np.arange(hor + 1)
for name, arr in irfs.items():
    med = np.median(arr[:, :, response, shock], axis=0)
    ax.plot(steps, med, lw=1.8, color=colors[name], label=name)
lo = np.quantile(irfs["sin tratamiento"][:, :, response, shock], 0.16, axis=0)
hi = np.quantile(irfs["sin tratamiento"][:, :, response, shock], 0.84, axis=0)
ax.fill_between(steps, lo, hi, color=colors["sin tratamiento"], alpha=0.12, lw=0)
ax.axhline(0, color="gray", lw=0.7)
ax.set_title("Respuesta de los ingresos fiscales a un choque de precios de exportación",
             fontsize=10.5)
ax.set_xlabel("trimestres"); ax.set_ylabel("pp")
ax.legend(fontsize=8.5, frameon=False)
plt.show()

rows = {}
for name, arr in irfs.items():
    med = np.median(arr[:, :, response, shock], axis=0)
    peak = int(med.argmax())
    after = med[peak:]
    half = np.argmax(after < 0.5 * med.max())          # medida desde el pico
    rows[name] = {"impacto (h=0)": med[0], "máximo": med.max(),
                  "h del máximo": peak,
                  "acumulado 8 trim.": med[:9].sum(),
                  "vida media tras el pico": int(half) if half else np.nan}
pd.DataFrame(rows).T.round(2)

La tabla es el cierre del argumento. Tomando como referencia el modelo estimado hasta 2019Q4:

| | impacto | pico | acumulado 8 trim. |
|:-|:-:|:-:|:-:|
| hasta 2019Q4 | 2.21 | 4.74 en $h=2$ | 21.9 |
| sin tratamiento | **3.61** | 6.23 en $h=1$ | **28.2** |
| dummies | 1.73 | 4.82 en $h=2$ | 22.6 |
| Lenza-Primiceri | 1.79 | 4.63 en $h=2$ | 21.7 |

Sin tratar la pandemia, el impacto se **sobreestima en 63 %** y el pico se adelanta un trimestre. Los
dos tratamientos vuelven a la referencia. La respuesta no cambió porque la economía cambiara: cambió
porque dos trimestres de 2020 entraron a la estimación como si fueran dinámica normal.

El punto general: la pandemia no es solo un problema de bandas. Contamina también el objeto central,
y en la sesión 2 ese objeto es todo lo que vamos a mirar.

## Ejercicios

1. **Cambie la ventana** a 2020Q1-2020Q4. ¿Qué pasa con las escalas estimadas, con las bandas y con
   la respuesta al impulso?
2. **Descarte la pandemia**: con la posterior estimada hasta 2019Q4, pronostique desde el último dato
   con la densidad predictiva a mano del notebook B. Compare sus bandas con las de Lenza-Primiceri.
3. **Fije las escalas en uno** (`covid_scales=1`) y verifique que reproduce el modelo sin tratamiento.
   Es la comprobación de que el tratamiento hace lo que dice.
4. **Lea Cascaldi-Garcia (2022)**: su prior sobre los dummies tiene una precisión $\kappa$. ¿Cuál de
   nuestros modelos corresponde a $\kappa \to 0$ y cuál a $\kappa \to \infty$?
5. **Pronostique desde dentro de la pandemia**: reestime con datos hasta 2021Q4 y proyecte ocho
   trimestres con los tres tratamientos. Aquí sí deberían separarse dummies y Lenza-Primiceri.
   ¿Por qué?